# Qwen3-TTS WebUI — Google Colab (Gradio版)

Google Colab 環境で動作する Gradio 版の Qwen3-TTS WebUI です。

## 使い方
1. **ランタイム → ランタイムのタイプを変更 → T4 GPU** を選択してから実行してください。
2. 各セルを順番に実行してください（`Shift+Enter`）。
3. 最後のセルを実行すると Gradio の公開 URL が表示されます。

## タブ説明
| タブ | 説明 |
|------|------|
| 🎓 ボイスモデル学習 | リファレンス音声からボイスモデルを作成・管理 |
| 🔊 音声合成 | 保存済みボイスモデルでテキストを音声化 |
| ⚡ 即時音声合成 | ボイスモデルを保存せず直接音声合成 |

In [ ]:
# @title セットアップ: 必要なパッケージのインストール
# Google Colab では PyTorch (CUDA版) がプリインストール済みです。
# Gradio / qwen-tts / soundfile のみ追加インストールします。
import subprocess, sys, importlib

pkgs = {
    "gradio": "gradio>=4.0.0",
    "qwen_tts": "qwen-tts",
    "soundfile": "soundfile",
}

print("パッケージをインストール中...")
for module_name, install_spec in pkgs.items():
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", install_spec], check=True)
    version = importlib.import_module(module_name).__version__ if hasattr(importlib.import_module(module_name), "__version__") else "installed"
    print(f"  ✅ {install_spec.split('>=')[0]} ({version})")

import torch
print(f"  ✅ torch ({torch.__version__}) - CUDA: {torch.cuda.is_available()}")
print("インストール完了")


In [ ]:
# @title エンジン & ボイスストア定義
import io
import json
import os
import tempfile
from datetime import datetime
from pathlib import Path

import numpy as np
import soundfile as sf
import torch

# ----------------------------------------------------------
# Engine (from engine.py)
# ----------------------------------------------------------

SUPPORTED_LANGUAGES = [
    "Japanese", "Chinese", "English", "Korean",
    "German", "French", "Russian", "Portuguese",
    "Spanish", "Italian",
]

GREETINGS = {
    "Japanese": "こんにちは、はじめまして。よろしくお願いします。",
    "Chinese": "你好，很高兴认识你。请多多关照。",
    "English": "Hello, nice to meet you. How are you doing today?",
    "Korean": "안녕하세요, 만나서 반갑습니다. 잘 부탁드립니다.",
    "German": "Hallo, freut mich, Sie kennenzulernen. Wie geht es Ihnen?",
    "French": "Bonjour, enchanté de vous rencontrer. Comment allez-vous?",
    "Russian": "Здравствуйте, приятно познакомиться. Как у вас дела?",
    "Portuguese": "Olá, prazer em conhecê-lo. Como você está?",
    "Spanish": "Hola, mucho gusto en conocerte. ¿Cómo estás?",
    "Italian": "Ciao, piacere di conoscerti. Come stai?",
}

MODEL_IDS = {
    "1.7B": "Qwen/Qwen3-TTS-12Hz-1.7B-Base",
    "0.6B": "Qwen/Qwen3-TTS-12Hz-0.6B-Base",
}


def get_device() -> str:
    """利用可能なデバイスを検出する。CUDA > CPU の優先順位。"""
    if os.environ.get("FORCE_CPU", "").lower() in ("1", "true", "yes"):
        return "cpu"
    if torch.cuda.is_available():
        return "cuda:0"
    return "cpu"


def get_dtype(device: str) -> torch.dtype:
    """デバイスに適した dtype を返す。"""
    if "cuda" in device:
        return torch.bfloat16
    return torch.float32


def get_attn_implementation(device: str) -> str:
    """利用可能な attention 実装を返す。"""
    if "cuda" in device:
        try:
            import flash_attn  # noqa: F401
            return "flash_attention_2"
        except ImportError:
            return "sdpa"
    return "sdpa"


class TTSEngine:
    """Qwen3-TTS モデルのラッパー。モデルの遅延読み込みとキャッシュを行う。"""

    def __init__(self):
        self._model = None
        self._model_size: str | None = None
        self._device: str | None = None

    def _ensure_model(self, model_size: str = "1.7B"):
        """必要に応じてモデルをロードする。"""
        if self._model is not None and self._model_size == model_size:
            return
        from qwen_tts import Qwen3TTSModel

        device = get_device()
        dtype = get_dtype(device)
        attn_impl = get_attn_implementation(device)
        model_id = MODEL_IDS[model_size]
        print(f"モデルをロード中: {model_id} ({device}, {dtype}) ...")
        self._model = Qwen3TTSModel.from_pretrained(
            model_id,
            device_map=device,
            dtype=dtype,
            attn_implementation=attn_impl,
        )
        self._model_size = model_size
        self._device = device
        print("モデルのロード完了")

    def create_voice_prompt(self, ref_audio, ref_text: str, model_size: str = "1.7B"):
        """リファレンス音声からボイスクローンプロンプトを作成する。"""
        self._ensure_model(model_size)
        return self._model.create_voice_clone_prompt(
            ref_audio=ref_audio,
            ref_text=ref_text,
        )

    def generate_speech(
        self,
        text: str,
        language: str,
        voice_clone_prompt,
        model_size: str = "1.7B",
        temperature: float = 0.65,
        repetition_penalty: float = 1.15,
        top_p: float = 0.9,
        top_k: int = 50,
    ) -> tuple:
        """保存済みボイスクローンプロンプトを使用して音声を合成する。"""
        self._ensure_model(model_size)
        wavs, sr = self._model.generate_voice_clone(
            text=text,
            language=language,
            voice_clone_prompt=voice_clone_prompt,
            temperature=temperature,
            repetition_penalty=repetition_penalty,
            top_p=top_p,
            top_k=top_k,
        )
        return wavs[0], sr

    def generate_speech_direct(
        self,
        text: str,
        language: str,
        ref_audio,
        ref_text: str,
        model_size: str = "1.7B",
        temperature: float = 0.65,
        repetition_penalty: float = 1.15,
        top_p: float = 0.9,
        top_k: int = 50,
    ) -> tuple:
        """リファレンス音声から直接音声を合成する（プロンプト未保存）。"""
        self._ensure_model(model_size)
        wavs, sr = self._model.generate_voice_clone(
            text=text,
            language=language,
            ref_audio=ref_audio,
            ref_text=ref_text,
            temperature=temperature,
            repetition_penalty=repetition_penalty,
            top_p=top_p,
            top_k=top_k,
        )
        return wavs[0], sr

    def get_device_info(self) -> str:
        """現在のデバイス情報を返す。"""
        device = get_device()
        if "cuda" in device:
            name = torch.cuda.get_device_name(0)
            return f"GPU: {name}"
        return "CPU"


# ----------------------------------------------------------
# Voice Store (from voice_store.py)
# ----------------------------------------------------------

STORE_DIR = Path("/content/voice_models")
MAX_CACHED = 10


def _get_model_dir(model_size: str) -> Path:
    return STORE_DIR / model_size


def _get_metadata_file(model_size: str) -> Path:
    return _get_model_dir(model_size) / "metadata.json"


def _ensure_store(model_size: str):
    model_dir = _get_model_dir(model_size)
    model_dir.mkdir(parents=True, exist_ok=True)
    metadata_file = _get_metadata_file(model_size)
    if not metadata_file.exists():
        metadata_file.write_text("{}", encoding="utf-8")


def _load_metadata(model_size: str) -> dict:
    _ensure_store(model_size)
    return json.loads(_get_metadata_file(model_size).read_text(encoding="utf-8"))


def _save_metadata(model_size: str, metadata: dict):
    _ensure_store(model_size)
    _get_metadata_file(model_size).write_text(
        json.dumps(metadata, ensure_ascii=False, indent=2), encoding="utf-8"
    )


def _get_model_size_from_data(data: bytes) -> str | None:
    """pt ファイルのバイナリデータから model_size を推定する。"""
    try:
        loaded = torch.load(io.BytesIO(data), weights_only=False, map_location="cpu")
        if isinstance(loaded, dict) and "model_size" in loaded:
            return loaded["model_size"]
        return None
    except Exception:
        return None


def save_voice(nickname: str, prompt_items, language: str, model_size: str):
    """ボイスクローンプロンプトをニックネーム付きで保存する。"""
    _ensure_store(model_size)
    metadata = _load_metadata(model_size)
    if nickname not in metadata:
        if len(metadata) >= MAX_CACHED:
            if metadata:
                oldest = min(metadata, key=lambda k: metadata[k]["created_at"])
                remove_voice(oldest, model_size)
                metadata = _load_metadata(model_size)
    filepath = _get_model_dir(model_size) / f"{nickname}.pt"
    torch.save(prompt_items, filepath)
    metadata[nickname] = {
        "file": str(filepath),
        "language": language,
        "model_size": model_size,
        "created_at": datetime.now().isoformat(),
    }
    _save_metadata(model_size, metadata)


def load_voice(nickname: str, model_size: str):
    """保存済みボイスクローンプロンプトを読み込む。"""
    metadata = _load_metadata(model_size)
    if nickname not in metadata:
        raise ValueError(f"ボイスモデル '{nickname}' が見つかりません（{model_size}）")
    filepath = Path(metadata[nickname]["file"])
    return torch.load(filepath, weights_only=False)


def list_voices_by_size(model_size: str) -> list:
    """指定された model_size の保存済みボイスモデル一覧を返す（新しい順）。"""
    metadata = _load_metadata(model_size)
    voices = []
    for name, info in sorted(
        metadata.items(), key=lambda x: x[1]["created_at"], reverse=True
    ):
        voices.append({"nickname": name, **info})
    return voices


def remove_voice(nickname: str, model_size: str):
    """保存済みボイスモデルを削除する。"""
    metadata = _load_metadata(model_size)
    if nickname in metadata:
        filepath = Path(metadata[nickname]["file"])
        if filepath.exists():
            filepath.unlink()
        del metadata[nickname]
        _save_metadata(model_size, metadata)


def export_voice(nickname: str, model_size: str) -> tuple:
    """ボイスモデルをバイト列としてエクスポートする。"""
    metadata = _load_metadata(model_size)
    if nickname not in metadata:
        raise ValueError(f"ボイスモデル '{nickname}' が見つかりません（{model_size}）")
    filepath = Path(metadata[nickname]["file"])
    filename = f"{nickname}_{model_size}.pt"
    return filepath.read_bytes(), filename


def import_voice(nickname: str, data: bytes, language: str, model_size: str):
    """バイト列からボイスモデルをインポートする。"""
    _ensure_store(model_size)
    saved_model_size = _get_model_size_from_data(data)
    if saved_model_size and saved_model_size != model_size:
        raise ValueError(
            f"モデルサイズが一致しません。データ: {saved_model_size}, 指定: {model_size}"
        )
    filepath = _get_model_dir(model_size) / f"{nickname}.pt"
    filepath.write_bytes(data)
    metadata = _load_metadata(model_size)
    if nickname not in metadata:
        if len(metadata) >= MAX_CACHED:
            if metadata:
                oldest = min(metadata, key=lambda k: metadata[k]["created_at"])
                remove_voice(oldest, model_size)
                metadata = _load_metadata(model_size)
    metadata[nickname] = {
        "file": str(filepath),
        "language": language,
        "model_size": model_size,
        "created_at": datetime.now().isoformat(),
    }
    _save_metadata(model_size, metadata)


# ----------------------------------------------------------
# Global engine instance
# ----------------------------------------------------------
_engine = TTSEngine()

VOICE_LEARNING_MIN_DURATION: float = 3.0
VOICE_LEARNING_MAX_DURATION: float = 15.0


def validate_audio_duration(audio_path: str) -> str | None:
    """音声の長さを検証し、エラーメッセージを返す (OK 時は None)。"""
    try:
        with sf.SoundFile(audio_path) as f:
            duration = len(f) / f.samplerate
    except Exception as e:
        return f"音声ファイルの読み込みに失敗しました: {e}"
    if duration < VOICE_LEARNING_MIN_DURATION or duration > VOICE_LEARNING_MAX_DURATION:
        return (
            f"音声の長さ ({duration:.1f}秒) は "
            f"{VOICE_LEARNING_MIN_DURATION}〜{VOICE_LEARNING_MAX_DURATION} 秒の範囲内である必要があります。"
        )
    return None


def get_voice_names(model_size: str) -> list:
    return [v["nickname"] for v in list_voices_by_size(model_size)]


def voices_to_table(model_size: str) -> list:
    voices = list_voices_by_size(model_size)
    return [
        [v["nickname"], v["language"], v["model_size"], v["created_at"][:10]]
        for v in voices
    ]


print(f"✅ エンジン & ボイスストア定義完了")
print(f"   デバイス: {_engine.get_device_info()}")


In [ ]:
# @title Gradio ロジック関数定義
import gradio as gr


def fn_create_voice(audio_file, ref_text, ref_lang, nickname, model_size):
    """ボイスモデルを作成して保存する。"""
    if audio_file is None:
        return "❌ 音声ファイルをアップロードしてください。", gr.update(), gr.update(), gr.update()
    if not ref_text.strip():
        return "❌ リファレンステキストを入力してください。", gr.update(), gr.update(), gr.update()
    if not nickname.strip():
        return "❌ ニックネームを入力してください。", gr.update(), gr.update(), gr.update()
    err = validate_audio_duration(audio_file)
    if err:
        return f"❌ {err}", gr.update(), gr.update(), gr.update()
    try:
        prompt_items = _engine.create_voice_prompt(
            ref_audio=audio_file,
            ref_text=ref_text.strip(),
            model_size=model_size,
        )
        save_voice(nickname.strip(), prompt_items, ref_lang, model_size)
        names = get_voice_names(model_size)
        first = names[0] if names else None
        return (
            f"✅ ボイスモデル '{nickname.strip()}' を作成しました。",
            gr.update(choices=names, value=first),
            gr.update(choices=names, value=first),
            gr.update(value=voices_to_table(model_size)),
        )
    except Exception as e:
        return f"❌ エラー: {e}", gr.update(), gr.update(), gr.update()


def fn_preview_voice(audio_file, ref_text, ref_lang, model_size):
    """ボイスモデルをプレビュー再生する（保存なし）。"""
    if audio_file is None:
        return "❌ 音声ファイルをアップロードしてください。", None
    if not ref_text.strip():
        return "❌ リファレンステキストを入力してください。", None
    err = validate_audio_duration(audio_file)
    if err:
        return f"❌ {err}", None
    greeting = GREETINGS.get(ref_lang, GREETINGS["English"])
    try:
        prompt_items = _engine.create_voice_prompt(
            ref_audio=audio_file,
            ref_text=ref_text.strip(),
            model_size=model_size,
        )
        wav, sr = _engine.generate_speech(
            text=greeting,
            language=ref_lang,
            voice_clone_prompt=prompt_items,
            model_size=model_size,
        )
        return f"✅ プレビュー生成完了: 「{greeting}」", (sr, wav)
    except Exception as e:
        return f"❌ エラー: {e}", None


def fn_refresh_voices(model_size):
    """保存済みボイスモデルの一覧を更新する。"""
    names = get_voice_names(model_size)
    data = voices_to_table(model_size)
    first = names[0] if names else None
    return (
        gr.update(choices=names, value=first),
        gr.update(choices=names, value=first),
        gr.update(value=data),
        gr.update(choices=names, value=first),
    )


def fn_delete_voice(nickname, model_size):
    """ボイスモデルを削除する。"""
    if not nickname:
        return "❌ 削除するモデルを選択してください。", gr.update(), gr.update(), gr.update(), gr.update()
    try:
        remove_voice(nickname, model_size)
        names = get_voice_names(model_size)
        first = names[0] if names else None
        return (
            f"✅ ボイスモデル '{nickname}' を削除しました。",
            gr.update(choices=names, value=first),
            gr.update(choices=names, value=first),
            gr.update(value=voices_to_table(model_size)),
            gr.update(choices=names, value=first),
        )
    except Exception as e:
        return f"❌ エラー: {e}", gr.update(), gr.update(), gr.update(), gr.update()


def fn_export_voice(nickname, model_size):
    """ボイスモデルをエクスポートしてダウンロード用ファイルを返す。"""
    if not nickname:
        return "❌ エクスポートするモデルを選択してください。", None
    try:
        data, filename = export_voice(nickname, model_size)
        tmp = tempfile.NamedTemporaryFile(delete=False, suffix=".pt")
        tmp.write(data)
        tmp.close()
        return f"✅ エクスポート準備完了: {filename}", tmp.name
    except Exception as e:
        return f"❌ エラー: {e}", None


def fn_import_voice(import_file, import_name, import_lang, model_size):
    """ボイスモデルをインポートする。"""
    if import_file is None:
        return "❌ .pt ファイルをアップロードしてください。", gr.update(), gr.update(), gr.update(), gr.update()
    if not import_name.strip():
        return "❌ インポート名を入力してください。", gr.update(), gr.update(), gr.update(), gr.update()
    try:
        with open(import_file, "rb") as f:
            data = f.read()
        import_voice(import_name.strip(), data, import_lang, model_size)
        names = get_voice_names(model_size)
        first = names[0] if names else None
        return (
            f"✅ ボイスモデル '{import_name.strip()}' をインポートしました。",
            gr.update(choices=names, value=first),
            gr.update(choices=names, value=first),
            gr.update(value=voices_to_table(model_size)),
            gr.update(choices=names, value=first),
        )
    except Exception as e:
        return f"❌ エラー: {e}", gr.update(), gr.update(), gr.update(), gr.update()


def fn_synthesize_tts(
    voice_name, output_lang, text, model_size,
    temperature, top_p, top_k, rep_penalty,
):
    """保存済みボイスモデルを使用して音声を合成する。"""
    if not voice_name:
        return "❌ ボイスモデルを選択してください。", None
    if not text.strip():
        return "❌ テキストを入力してください。", None
    try:
        voice_prompt = load_voice(voice_name, model_size)
        wav, sr = _engine.generate_speech(
            text=text.strip(),
            language=output_lang,
            voice_clone_prompt=voice_prompt,
            model_size=model_size,
            temperature=temperature,
            repetition_penalty=rep_penalty,
            top_p=top_p,
            top_k=int(top_k),
        )
        return "✅ 音声合成完了", (sr, wav)
    except Exception as e:
        return f"❌ エラー: {e}", None


def fn_synthesize_instant(
    audio_file, ref_text, ref_lang, output_lang, text, model_size,
    temperature, top_p, top_k, rep_penalty,
):
    """リファレンス音声から直接音声を合成する（保存なし）。"""
    if audio_file is None:
        return "❌ 音声ファイルをアップロードしてください。", None
    if not ref_text.strip():
        return "❌ リファレンステキストを入力してください。", None
    if not text.strip():
        return "❌ 合成テキストを入力してください。", None
    err = validate_audio_duration(audio_file)
    if err:
        return f"❌ {err}", None
    try:
        wav, sr = _engine.generate_speech_direct(
            text=text.strip(),
            language=output_lang,
            ref_audio=audio_file,
            ref_text=ref_text.strip(),
            model_size=model_size,
            temperature=temperature,
            repetition_penalty=rep_penalty,
            top_p=top_p,
            top_k=int(top_k),
        )
        return "✅ 音声合成完了", (sr, wav)
    except Exception as e:
        return f"❌ エラー: {e}", None


def on_model_size_change(model_size):
    """モデルサイズ変更時に全ドロップダウンを更新する。"""
    names = get_voice_names(model_size)
    data = voices_to_table(model_size)
    first = names[0] if names else None
    return (
        gr.update(choices=names, value=first),  # train_delete_dd
        gr.update(choices=names, value=first),  # train_export_dd
        gr.update(value=data),                  # train_voice_table
        gr.update(choices=names, value=first),  # tts_voice_dd
    )


print("✅ Gradio ロジック関数定義完了")


In [ ]:
# @title Gradio UI 構築 & 起動
# Google Colab では share=True で公開 URL が生成されます。

_initial_size = "1.7B"
_init_names = get_voice_names(_initial_size)
_init_data = voices_to_table(_initial_size)

with gr.Blocks(title="Qwen3-TTS WebUI (Google Colab)", theme=gr.themes.Soft()) as demo:

    gr.Markdown(
        f"# 🎤 Qwen3-TTS WebUI\n"
        f"**デバイス**: {_engine.get_device_info()}　|　"
        "モデルは最初の音声合成時に自動ロードされます。"
    )

    model_size_radio = gr.Radio(
        label="モデルサイズ (全タブ共通)",
        choices=["1.7B", "0.6B"],
        value="1.7B",
        info="1.7B = 高品質、0.6B = 軽量・高速",
    )

    with gr.Tabs():

        # ================================================================
        # Tab 1: ボイスモデル学習
        # ================================================================
        with gr.Tab("🎓 ボイスモデル学習"):
            gr.Markdown(
                "リファレンス音声 (3〜15秒) からオリジナルボイスモデルを作成・保存します。"
            )

            with gr.Row():
                # --- 左カラム: 新規作成 ---
                with gr.Column(scale=3):
                    gr.Markdown("### 新しいモデルを作成")
                    train_audio = gr.Audio(
                        label="リファレンス音声 (3〜15秒)",
                        type="filepath",
                    )
                    train_ref_text = gr.Textbox(
                        label="リファレンステキスト（音声の文字起こし）",
                        placeholder="音声の内容をそのまま入力してください...",
                        lines=3,
                    )
                    with gr.Row():
                        train_ref_lang = gr.Dropdown(
                            label="音声の言語",
                            choices=SUPPORTED_LANGUAGES,
                            value="Japanese",
                        )
                        train_nickname = gr.Textbox(
                            label="ニックネーム",
                            placeholder="my_voice",
                        )
                    with gr.Row():
                        train_create_btn = gr.Button("🎯 モデル作成", variant="primary")
                        train_preview_btn = gr.Button("▶️ プレビュー再生")
                    train_status = gr.Textbox(label="ステータス", interactive=False, lines=2)
                    train_preview_audio = gr.Audio(label="プレビュー音声", type="numpy")

                # --- 右カラム: 保存済みモデル管理 ---
                with gr.Column(scale=2):
                    gr.Markdown("### 保存済みモデル")
                    train_refresh_btn = gr.Button("🔄 一覧を更新")
                    train_voice_table = gr.Dataframe(
                        value=_init_data,
                        headers=["ニックネーム", "言語", "サイズ", "作成日"],
                        interactive=False,
                        wrap=True,
                    )

                    gr.Markdown("**削除**")
                    train_delete_dd = gr.Dropdown(
                        label="削除するモデル",
                        choices=_init_names,
                        value=_init_names[0] if _init_names else None,
                    )
                    train_delete_btn = gr.Button("🗑️ 削除", variant="stop")

                    gr.Markdown("**エクスポート**")
                    train_export_dd = gr.Dropdown(
                        label="エクスポートするモデル",
                        choices=_init_names,
                        value=_init_names[0] if _init_names else None,
                    )
                    train_export_btn = gr.Button("📤 エクスポート (.pt)")
                    train_export_file = gr.File(label="ダウンロード")

                    gr.Markdown("**インポート**")
                    train_import_file = gr.File(
                        label=".pt ファイルをアップロード",
                        file_types=[".pt"],
                    )
                    train_import_name = gr.Textbox(
                        label="インポート名",
                        placeholder="imported_voice",
                    )
                    train_import_lang = gr.Dropdown(
                        label="言語",
                        choices=SUPPORTED_LANGUAGES,
                        value="Japanese",
                    )
                    train_import_btn = gr.Button("📥 インポート")

        # ================================================================
        # Tab 2: 音声合成（保存済みボイス使用）
        # ================================================================
        with gr.Tab("🔊 音声合成"):
            gr.Markdown("保存済みボイスモデルを使用してテキストを音声に変換します。")
            with gr.Row():
                with gr.Column(scale=2):
                    gr.Markdown("#### ボイスモデル設定")
                    tts_voice_dd = gr.Dropdown(
                        label="ボイスモデル",
                        choices=_init_names,
                        value=_init_names[0] if _init_names else None,
                    )
                    tts_refresh_btn = gr.Button("🔄 ボイスモデル一覧を更新")
                    tts_output_lang = gr.Dropdown(
                        label="出力言語",
                        choices=SUPPORTED_LANGUAGES,
                        value="Japanese",
                    )
                    with gr.Accordion("生成パラメータ", open=False):
                        tts_temperature = gr.Slider(
                            0.30, 1.30, value=0.65, step=0.05,
                            label="温度（感情値）",
                            info="高いほど表現豊か、低いほど安定した発音",
                        )
                        tts_top_p = gr.Slider(
                            0.80, 1.00, value=0.90, step=0.05,
                            label="Top-p（核サンプリング）",
                        )
                        tts_top_k = gr.Slider(
                            10, 50, value=50, step=1,
                            label="Top-k",
                        )
                        tts_rep_penalty = gr.Slider(
                            1.00, 1.50, value=1.15, step=0.05,
                            label="繰り返し抑制（Repetition Penalty）",
                        )

                with gr.Column(scale=3):
                    gr.Markdown("#### テキスト入力")
                    tts_text = gr.Textbox(
                        label="合成テキスト",
                        placeholder="ここに読み上げさせたいテキストを入力してください...",
                        lines=6,
                    )
                    tts_generate_btn = gr.Button("🎵 音声合成", variant="primary")
                    tts_status = gr.Textbox(label="ステータス", interactive=False)
                    tts_audio_out = gr.Audio(label="生成音声", type="numpy")

        # ================================================================
        # Tab 3: 即時音声合成（ボイスモデル保存なし）
        # ================================================================
        with gr.Tab("⚡ 即時音声合成"):
            gr.Markdown(
                "ボイスモデルを保存せずに、リファレンス音声から直接音声を合成します。"
            )
            with gr.Row():
                with gr.Column():
                    gr.Markdown("#### リファレンス音声")
                    instant_audio = gr.Audio(
                        label="リファレンス音声 (3〜15秒)",
                        type="filepath",
                    )
                    instant_ref_text = gr.Textbox(
                        label="リファレンステキスト（音声の文字起こし）",
                        placeholder="音声の内容をそのまま入力してください...",
                        lines=3,
                    )
                    instant_ref_lang = gr.Dropdown(
                        label="音声の言語",
                        choices=SUPPORTED_LANGUAGES,
                        value="Japanese",
                    )

                with gr.Column():
                    gr.Markdown("#### 音声合成設定")
                    instant_output_lang = gr.Dropdown(
                        label="出力言語",
                        choices=SUPPORTED_LANGUAGES,
                        value="Japanese",
                    )
                    instant_text = gr.Textbox(
                        label="合成テキスト",
                        placeholder="ここに読み上げさせたいテキストを入力してください...",
                        lines=5,
                    )
                    with gr.Accordion("生成パラメータ", open=False):
                        instant_temperature = gr.Slider(
                            0.30, 1.30, value=0.65, step=0.05,
                            label="温度（感情値）",
                        )
                        instant_top_p = gr.Slider(
                            0.80, 1.00, value=0.90, step=0.05,
                            label="Top-p",
                        )
                        instant_top_k = gr.Slider(
                            10, 50, value=50, step=1,
                            label="Top-k",
                        )
                        instant_rep_penalty = gr.Slider(
                            1.00, 1.50, value=1.15, step=0.05,
                            label="繰り返し抑制",
                        )
                    instant_generate_btn = gr.Button("🎵 音声合成", variant="primary")

            instant_status = gr.Textbox(label="ステータス", interactive=False)
            instant_audio_out = gr.Audio(label="生成音声", type="numpy")

    # ================================================================
    # イベントハンドラー
    # ================================================================

    # モデルサイズ変更 → 全ドロップダウン更新
    model_size_radio.change(
        fn=on_model_size_change,
        inputs=[model_size_radio],
        outputs=[train_delete_dd, train_export_dd, train_voice_table, tts_voice_dd],
    )

    # --- ボイスモデル学習タブ ---
    train_create_btn.click(
        fn=fn_create_voice,
        inputs=[train_audio, train_ref_text, train_ref_lang, train_nickname, model_size_radio],
        outputs=[train_status, train_delete_dd, train_export_dd, train_voice_table],
    )

    train_preview_btn.click(
        fn=fn_preview_voice,
        inputs=[train_audio, train_ref_text, train_ref_lang, model_size_radio],
        outputs=[train_status, train_preview_audio],
    )

    train_refresh_btn.click(
        fn=fn_refresh_voices,
        inputs=[model_size_radio],
        outputs=[train_delete_dd, train_export_dd, train_voice_table, tts_voice_dd],
    )

    train_delete_btn.click(
        fn=fn_delete_voice,
        inputs=[train_delete_dd, model_size_radio],
        outputs=[train_status, train_delete_dd, train_export_dd, train_voice_table, tts_voice_dd],
    )

    train_export_btn.click(
        fn=fn_export_voice,
        inputs=[train_export_dd, model_size_radio],
        outputs=[train_status, train_export_file],
    )

    train_import_btn.click(
        fn=fn_import_voice,
        inputs=[train_import_file, train_import_name, train_import_lang, model_size_radio],
        outputs=[train_status, train_delete_dd, train_export_dd, train_voice_table, tts_voice_dd],
    )

    # --- 音声合成タブ ---
    tts_refresh_btn.click(
        fn=lambda ms: gr.update(
            choices=get_voice_names(ms),
            value=(get_voice_names(ms) or [None])[0],
        ),
        inputs=[model_size_radio],
        outputs=[tts_voice_dd],
    )

    tts_generate_btn.click(
        fn=fn_synthesize_tts,
        inputs=[
            tts_voice_dd, tts_output_lang, tts_text, model_size_radio,
            tts_temperature, tts_top_p, tts_top_k, tts_rep_penalty,
        ],
        outputs=[tts_status, tts_audio_out],
    )

    # --- 即時音声合成タブ ---
    instant_generate_btn.click(
        fn=fn_synthesize_instant,
        inputs=[
            instant_audio, instant_ref_text, instant_ref_lang,
            instant_output_lang, instant_text, model_size_radio,
            instant_temperature, instant_top_p, instant_top_k, instant_rep_penalty,
        ],
        outputs=[instant_status, instant_audio_out],
    )


# Google Colab 向け: share=True で公開 URL を取得
demo.launch(share=True, debug=False)
